In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches

In [ ]:
# Define the GridWorld environment
grid_size = (4, 4)
terminal_states = [(0, 0), (3, 3)]
rewards = -0.1 * np.ones(grid_size)
rewards[0, 0] = 1.0
rewards[3, 3] = 1.0

# Actions: up, down, left, right
actions = ['U', 'D', 'L', 'R']
action_to_delta = {
    'U': (-1, 0),
    'D': (1, 0),
    'L': (0, -1),
    'R': (0, 1)
}

# Transition model: deterministic
def transition(state, action):
    if state in terminal_states:
        return state
    delta = action_to_delta[action]
    next_state = (state[0] + delta[0], state[1] + delta[1])
    if 0 <= next_state[0] < grid_size[0] and 0 <= next_state[1] < grid_size[1]:
        return next_state
    return state

# Initialize value function
V = np.zeros(grid_size)
gamma = 0.9
theta = 1e-4

# Value Iteration (Model-based)
def value_iteration():
    global V
    while True:
        delta = 0
        new_V = np.copy(V)
        for i in range(grid_size[0]):
            for j in range(grid_size[1]):
                state = (i, j)
                if state in terminal_states:
                    continue
                values = []
                for action in actions:
                    next_state = transition(state, action)
                    reward = rewards[next_state]
                    values.append(reward + gamma * V[next_state])
                best_value = max(values)
                delta = max(delta, np.abs(best_value - V[state]))
                new_V[state] = best_value
        V = new_V
        if delta < theta:
            break

value_iteration()

# Derive policy from value function
policy = np.full(grid_size, '', dtype=object)
for i in range(grid_size[0]):
    for j in range(grid_size[1]):
        state = (i, j)
        if state in terminal_states:
            policy[state] = 'T'
            continue
        values = []
        for action in actions:
            next_state = transition(state, action)
            reward = rewards[next_state]
            values.append((reward + gamma * V[next_state], action))
        best_action = max(values, key=lambda x: x[0])[1]
        policy[state] = best_action

# Plot the grid with values and policy
fig, ax = plt.subplots(figsize=(6, 6))
for i in range(grid_size[0]):
    for j in range(grid_size[1]):
        ax.add_patch(patches.Rectangle((j, grid_size[0]-1-i), 1, 1, fill=False))
        ax.text(j + 0.5, grid_size[0]-1-i + 0.7, f"{V[i, j]:.2f}", ha='center', fontsize=8)
        ax.text(j + 0.5, grid_size[0]-1-i + 0.3, policy[i, j], ha='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, grid_size[1])
ax.set_ylim(0, grid_size[0])
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Value Function and Derived Policy (Model-based MDP)")
plt.grid(True)
plt.tight_layout()
plt.show()
